# Day 2: From Brownian Motion to Rough Volatility — The Texture of τ

*Alpha Flow Research · HongJin HE · July 2026*

---

## Recap: Two Kinds of Randomness

In Day 1, we saw that factor models fail because markets are *reflexive* — the map changes the territory.
Today we go deeper: even if you accept that markets are random, **not all randomness is the same kind**.

Standard Black-Scholes assumes log-returns are driven by a single Brownian motion $W_t$:
$$dS_t = \mu S_t dt + \sigma S_t dW_t$$

This is clean, tractable — and empirically wrong in three ways:
1. **Volatility is itself stochastic** (not constant σ)
2. **Volatility has memory** — today's vol predicts tomorrow's
3. **Jumps exist** — discontinuous moves that no Brownian path can produce

This notebook establishes our decomposition of market noise into **τ** (physical/continuous) and **η** (behavioral/jump).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy import stats

np.random.seed(2026)
T, N = 252, 10000  # 1 trading year, fine grid
dt = T / N
t = np.linspace(0, T, N)

# ── 1. Standard Brownian Motion ──────────────────────────────────────────
dW = np.random.normal(0, np.sqrt(dt), N)
W = np.cumsum(dW)

# ── 2. Rough Fractional Brownian Motion (H ≈ 0.1 for volatility) ────────
# Simple Cholesky approximation for illustration
def fBm_davies_harte(N, H):
    """Davies-Harte method for fractional Brownian motion."""
    k = np.arange(N)
    cov_row = 0.5 * (np.abs(k+1)**(2*H) - 2*np.abs(k)**(2*H) + np.abs(k-1)**(2*H))
    cov_row[0] = 1.0
    # Embed in circulant
    c = np.concatenate([cov_row, cov_row[-1:0:-1]])
    lam = np.real(np.fft.fft(c))
    lam = np.maximum(lam, 0)  # numerical safeguard
    W_tilde = np.fft.fft(np.sqrt(lam) * (np.random.randn(2*N) + 1j*np.random.randn(2*N)) / np.sqrt(2*N))
    return np.real(W_tilde[:N]) * N**H

W_rough = fBm_davies_harte(N, H=0.1)  # H=0.1 → very rough (Gatheral et al. 2018)
W_smooth = fBm_davies_harte(N, H=0.7)  # H=0.7 → persistent/smooth

print(f'Standard BM Hurst (empirical): {np.polyfit(np.log(np.arange(1,50)), np.log([np.mean(np.abs(np.diff(W, n=k))) for k in range(1,50)]), 1)[0]:.3f}')
print(f'Expected for H=0.1: 0.1')
print(f'Expected for H=0.7: 0.7')

## The Bipower Variation Decomposition

The key insight from Barndorff-Nielsen & Shephard (2004) is that **quadratic variation** $[X]_t$ and **bipower variation** $BPV_t$ diverge when jumps are present:

$$QV_t = \sum_{i=1}^n r_{t_i}^2 \xrightarrow{p} \int_0^t \sigma_s^2 ds + \sum_{s \le t} \Delta X_s^2$$

$$BPV_t = \frac{\pi}{2} \sum_{i=2}^n |r_{t_i}||r_{t_{i-1}}| \xrightarrow{p} \int_0^t \sigma_s^2 ds$$

The difference $QV_t - BPV_t$ **isolates the jump component** (behavioral noise η).

This is the mathematical foundation for our dual-noise decomposition:
$$dZ_t = \underbrace{f(Z_t, \mu_t) dt + \sigma_\tau(Z_t) dW_t}_{\text{physical noise } \tau} + \underbrace{\int_{\mathbb{R}} g(Z_{t^-}, x) \tilde{N}(dt, dx)}_{\text{behavioral noise } \eta}$$

In [ ]:
# Simulate returns with both continuous and jump components
sigma_base = 0.015  # ~1.5% daily vol
jump_intensity = 5 / 252  # ~5 jumps per year
jump_size_mean = 0.0
jump_size_std = 0.04  # ~4% jump magnitude

n_days = 252 * 5  # 5 years
dt_day = 1

# Continuous component (τ): stochastic vol via GARCH(1,1)
omega, alpha_g, beta_g = 1e-5, 0.10, 0.85
vol = np.zeros(n_days)
vol[0] = sigma_base
eps = np.random.randn(n_days)

for t_i in range(1, n_days):
    vol[t_i] = np.sqrt(omega + alpha_g * (vol[t_i-1]*eps[t_i-1])**2 + beta_g * vol[t_i-1]**2)

r_continuous = vol * eps

# Jump component (η): compound Poisson
n_jumps = np.random.poisson(jump_intensity, n_days)
jump_sizes = np.array([np.sum(np.random.normal(jump_size_mean, jump_size_std, k)) for k in n_jumps])
r_total = r_continuous + jump_sizes

# Bipower Variation vs Quadratic Variation (rolling 21-day)
window = 21
QV = np.array([np.sum(r_total[i:i+window]**2) for i in range(n_days - window)])
BPV = np.array([(np.pi/2) * np.sum(np.abs(r_total[i:i+window-1]) * np.abs(r_total[i+1:i+window])) 
                for i in range(n_days - window)])

jump_component = np.maximum(QV - BPV, 0)  # ≥ 0 by construction
jump_ratio = jump_component / np.maximum(QV, 1e-10)  # fraction of variance from jumps

fig, axes = plt.subplots(3, 1, figsize=(12, 9))

days = np.arange(len(QV))
axes[0].plot(days, np.sqrt(252*BPV/window), label='Continuous vol (√BPV·252)', color='steelblue', lw=1.2)
axes[0].plot(days, np.sqrt(252*QV/window), label='Total vol (√QV·252)', color='coral', lw=1.2, alpha=0.7)
axes[0].set_ylabel('Annualised Vol')
axes[0].legend()
axes[0].set_title('Physical (τ) vs Total Noise: Bipower Variation Decomposition')

axes[1].fill_between(days, jump_ratio, alpha=0.6, color='orange', label='Jump share = η/(τ+η)')
axes[1].set_ylabel('Jump fraction')
axes[1].set_ylim(0, 0.8)
axes[1].legend()
axes[1].axhline(np.mean(jump_ratio), ls='--', color='red', label=f'Mean={np.mean(jump_ratio):.2f}')
axes[1].legend()

axes[2].hist(r_continuous, bins=60, alpha=0.6, density=True, label='Physical noise τ (Gaussian)')
axes[2].hist(r_total, bins=60, alpha=0.6, density=True, label='Total returns (fat-tailed)')
x_range = np.linspace(-0.15, 0.15, 300)
axes[2].plot(x_range, stats.norm.pdf(x_range, 0, sigma_base), 'b--', lw=2, label='N(0,σ²) reference')
axes[2].set_xlabel('Daily return')
axes[2].legend()

plt.tight_layout()
plt.savefig('../figures/dual_noise_bpv.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Average jump share of total variance: {np.mean(jump_ratio):.1%}')
print(f'Kurtosis of total returns: {stats.kurtosis(r_total):.2f} (Gaussian = 0)')

## The Cramér-Rao Lower Bound for Prediction

The dual-noise decomposition gives us a fundamental limit on prediction accuracy. By the Cramér-Rao bound:

$$\text{Var}(\hat{\mu}) \ge \underbrace{\frac{\sigma_\tau^2}{\Delta t}}_{\text{irreducible from }\tau} + \underbrace{\nu_\eta(\mathbb{R})}_{\text{irreducible from }\eta}$$

where $\nu_\eta$ is the Lévy measure of the jump component.

**Implication**: No matter how sophisticated your model, prediction error is bounded below by this sum. The two terms are:
- $\sigma_\tau^2 / \Delta t$: physical diffusion uncertainty (decreases with sampling frequency)
- $\nu_\eta(\mathbb{R})$: jump intensity (irreducible — you can't know when a CEO resigns)

**This is why alpha decays**: as sampling frequency increases, $\Delta t \to 0$ and the diffusion term dominates, washing out any predictable signal.

Tomorrow (Day 3) we'll see how the Encoder E learns to *separate* τ from η in latent space.

In [ ]:
# Cramér-Rao lower bound as function of sampling frequency
sigma_tau_sq = sigma_base**2
nu_eta = jump_intensity  # jumps/day

delta_t_range = np.logspace(-3, 0, 100)  # from millisecond to daily
cr_bound = sigma_tau_sq / delta_t_range + nu_eta

fig, ax = plt.subplots(figsize=(9, 5))
ax.loglog(delta_t_range, cr_bound, 'k-', lw=2, label='CR Lower Bound')
ax.loglog(delta_t_range, sigma_tau_sq / delta_t_range, 'b--', label='Diffusion term σ²_τ/Δt')
ax.axhline(nu_eta, color='red', ls='--', label=f'Jump floor ν_η = {nu_eta:.4f}')
ax.set_xlabel('Sampling interval Δt (days)')
ax.set_ylabel('Minimum prediction variance')
ax.set_title('Cramér-Rao Bound: Irreducible Prediction Error by Frequency')
ax.legend()
ax.axvline(1, color='gray', ls=':', alpha=0.5, label='Daily')
plt.tight_layout()
plt.savefig('../figures/cramer_rao_bound.png', dpi=150, bbox_inches='tight')
plt.show()
print('Key insight: HFT reduces diffusion error but cannot escape the jump floor.')
print('Long-horizon investors face growing diffusion uncertainty — no free lunch.')